In [ ]:
"""
Sell Bot — Position Monitor -> IBKR Paper Trading -> MongoDB
Enhanced with trailing stops, flash crash detection, and emergency exits.
"""

import logging
import sys
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone
import random
from typing import Optional
from dataclasses import dataclass

from pymongo import MongoClient
from ib_insync import *

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ─── IBKR Paper Trading ───────────────────────────────────────────────────────
IBKR_HOST = "127.0.0.1"
IBKR_PORT = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)

# ─── Trade Rules ──────────────────────────────────────────────────────────────
PROFIT_TARGET = 0.50           # Take profit at 50%
MAX_LOSS_PCT = 0.03            # HARD STOP: Never lose more than 3%
TRAILING_STOP_PCT = 0.05       # Give back max 5% from highs
FLASH_CRASH_PCT = 0.03         # Exit if drops 3% in single interval
LIQUIDITY_SPREAD_PCT = 0.005   # Exit if spread > 0.5%
MONITOR_INTERVAL = 30          # Check every 30 seconds
EMERGENCY_TIMEOUT = 5          # Seconds to wait for limit fill before market

# ─── MongoDB ──────────────────────────────────────────────────────────────────
mongo_client = MongoClient("mongodb://localhost:27017/")
db = mongo_client["brkout_database"]
trades_collection = db["trades"]


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING (UTF-8 safe)
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh = logging.FileHandler("sell_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("sell_bot")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

class IBKRSellClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()
        self._price_cache = {}  # symbol -> (price, timestamp)

    def connect(self):
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    def get_last_price(self, symbol: str) -> Optional[float]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        self.ib.sleep(3)
        self.ib.cancelMktData(contract)
        
        for label, val in [("last", ticker.last), ("ask", ticker.ask), ("bid", ticker.bid), ("close", ticker.close)]:
            if val and float(val) > 0:
                log.info(f"{symbol}: price source='{label}' value=${float(val):.4f}")
                self._price_cache[symbol] = (float(val), datetime.now(timezone.utc))
                return float(val)
        
        log.warning(f"{symbol}: no price available from IBKR")
        return None

    def get_ticker(self, symbol: str) -> Ticker:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "", False, False)
        self.ib.sleep(2)
        return ticker

    def get_bars(self, symbol: str, n: int = 70) -> Optional[pd.DataFrame]:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        
        for what_to_show in ("TRADES", "MIDPOINT"):
            raw = self.ib.reqHistoricalData(
                contract,
                endDateTime="",
                durationStr="10 D",
                barSizeSetting="5 mins",
                whatToShow=what_to_show,
                useRTH=False,
                keepUpToDate=False,
            )
            if raw:
                log.info(f"{symbol}: got {len(raw)} bars (whatToShow={what_to_show})")
                df = pd.DataFrame([
                    {"high": b.high, "low": b.low, "close": b.close, "volume": getattr(b, 'volume', 0)}
                    for b in raw
                ])
                return df.tail(n).reset_index(drop=True)
        
        log.warning(f"{symbol}: no bars returned from IBKR")
        return None

    def psar_exit_signal(self, symbol: str) -> bool:
        try:
            df = self.get_bars(symbol, n=70)
            if df is None or len(df) < 10:
                log.warning(f"{symbol}: not enough bars for PSAR")
                return False
            
            psar = self._compute_psar(df["high"].values, df["low"].values, df["close"].values)
            price = float(df["close"].iloc[-1])
            exit_signal = bool(psar[-1] > price)
            log.info(f"{symbol}: PSAR={psar[-1]:.4f} price={price:.4f} exit={exit_signal}")
            return exit_signal
        except Exception as e:
            log.error(f"{symbol}: PSAR error - {e}")
            return False

    @staticmethod
    def _compute_psar(highs, lows, closes, iaf=0.02, max_af=0.2) -> np.ndarray:
        n = len(closes)
        psar = closes.copy().astype(float)
        bull = True
        af = iaf
        hp = float(highs[0])
        lp = float(lows[0])
        
        for i in range(2, n):
            if bull:
                psar[i] = psar[i-1] + af * (hp - psar[i-1])
                psar[i] = min(psar[i], lows[i-1], lows[i-2])
                if lows[i] < psar[i]:
                    bull = False
                    psar[i] = hp
                    lp = lows[i]
                    af = iaf
                else:
                    if highs[i] > hp:
                        hp = highs[i]
                        af = min(af + iaf, max_af)
            else:
                psar[i] = psar[i-1] + af * (lp - psar[i-1])
                psar[i] = max(psar[i], highs[i-1], highs[i-2])
                if highs[i] > psar[i]:
                    bull = True
                    psar[i] = lp
                    hp = highs[i]
                    af = iaf
                else:
                    if lows[i] < lp:
                        lp = lows[i]
                        af = min(af + iaf, max_af)
        return psar

    def get_avg_fill_price_from_broker(self, symbol: str, order_id: int) -> Optional[float]:
        # Check open trades
        for trade in self.ib.trades():
            if (trade.contract.symbol == symbol 
                and trade.order.orderId == order_id 
                and trade.orderStatus.avgFillPrice > 0):
                avg = float(trade.orderStatus.avgFillPrice)
                log.info(f"{symbol}: broker avg fill (trades) = ${avg:.4f}")
                return avg

        # Check fills history
        fills = self.ib.fills()
        log.info(f"{symbol}: scanning {len(fills)} fill(s) for orderId={order_id}")
        matched = [f for f in fills if f.contract.symbol == symbol and f.execution.orderId == order_id]
        
        if matched:
            total_qty = sum(f.execution.shares for f in matched)
            total_cost = sum(f.execution.shares * f.execution.price for f in matched)
            avg = total_cost / total_qty
            log.info(f"{symbol}: fills avg fill = ${avg:.4f} (qty={total_qty})")
            return avg

        log.warning(f"{symbol}: no fill found for orderId={order_id}")
        return None

    def place_limit_sell(self, symbol: str, price: float, qty: int, emergency: bool = False) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        
        # For emergency, go 1% below bid for faster fill
        if emergency:
            price = price * 0.99
            
        order = LimitOrder("SELL", qty, round(price, 2))
        order.outsideRth = True
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)
        
        log.info(f"SELL LIMIT {symbol} x{qty} @ ${price:.2f} orderId={trade.order.orderId} emergency={emergency}")
        return trade

    def place_market_sell(self, symbol: str, qty: int) -> Trade:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        
        order = MarketOrder("SELL", qty)
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)
        
        log.info(f"SELL MARKET {symbol} x{qty} orderId={trade.order.orderId}")
        return trade

    def get_sell_fill_price(self, symbol: str, sell_order_id: int, fallback_price: float) -> float:
        self.ib.sleep(3)
        avg = self.get_avg_fill_price_from_broker(symbol, sell_order_id)
        if avg:
            return avg
        log.warning(f"{symbol}: sell fill not confirmed, using fallback ${fallback_price:.4f}")
        return fallback_price


# ══════════════════════════════════════════════════════════════════════════════
# RISK MANAGEMENT FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def update_trailing_stop(trade: dict, current_price: float, ibkr: IBKRSellClient) -> float:
    """
    Update highest price seen and calculate trailing stop level.
    Returns the effective stop price.
    """
    entry_price = float(trade.get("avg_fill_price") or trade["entry_price"])
    highest = trade.get("highest_price", entry_price)
    new_high = max(highest, current_price)
    
    # Calculate stops
    trailing_level = new_high * (1 - TRAILING_STOP_PCT)
    hard_stop = entry_price * (1 - MAX_LOSS_PCT)
    
    # Use whichever is higher (less risky)
    effective_stop = max(trailing_level, hard_stop)
    
    # Update DB if new high
    if new_high > highest:
        trades_collection.update_one(
            {"_id": trade["_id"]},
            {"$set": {
                "highest_price": new_high,
                "trailing_stop": effective_stop,
                "hard_stop": hard_stop
            }}
        )
        log.info(f"{trade['symbol']}: New high ${new_high:.4f}, trailing stop @ ${effective_stop:.4f}")
    
    return effective_stop


def check_flash_crash(symbol: str, current_price: float, trade: dict) -> Optional[str]:
    """Check for sudden price drops that indicate flash crash."""
    last_price = trade.get("last_price")
    last_time = trade.get("last_price_time")
    
    # Update current price in DB for next iteration
    trades_collection.update_one(
        {"_id": trade["_id"]},
        {"$set": {
            "last_price": current_price,
            "last_price_time": datetime.now(timezone.utc)
        }}
    )
    
    if last_price and last_price > 0:
        drop_pct = (current_price - last_price) / last_price
        
        if drop_pct <= -FLASH_CRASH_PCT:
            log.warning(f"{symbol}: FLASH CRASH detected! Drop {drop_pct*100:.2f}% in {MONITOR_INTERVAL}s")
            return f"flash_crash_{abs(drop_pct)*100:.1f}pct"
    
    return None


def check_liquidity_crisis(ibkr: IBKRSellClient, symbol: str) -> bool:
    """Detect if bid-ask spread is widening (liquidity drying up)."""
    try:
        ticker = ibkr.get_ticker(symbol)
        if ticker.bid and ticker.ask and ticker.bid > 0:
            mid = (ticker.ask + ticker.bid) / 2
            spread_pct = (ticker.ask - ticker.bid) / mid
            
            if spread_pct > LIQUIDITY_SPREAD_PCT:
                log.warning(f"{symbol}: Liquidity crisis! Spread {spread_pct*100:.2f}%")
                return True
    except Exception as e:
        log.error(f"{symbol}: Error checking liquidity - {e}")
    
    return False


def calculate_atr(ibkr: IBKRSellClient, symbol: str, period: int = 14) -> Optional[float]:
    """Calculate Average True Range for volatility assessment."""
    df = ibkr.get_bars(symbol, n=period + 10)
    if df is None or len(df) < period:
        return None
    
    highs, lows, closes = df["high"], df["low"], df["close"]
    
    tr1 = highs - lows
    tr2 = abs(highs - closes.shift(1))
    tr3 = abs(lows - closes.shift(1))
    
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(period).mean().iloc[-1]
    
    return float(atr) if pd.notna(atr) else None


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def get_open_trades() -> list:
    return list(trades_collection.find({"status": "open"}))


def get_effective_entry_price(trade: dict, ibkr: IBKRSellClient) -> float:
    """Priority: avg_fill_price (DB) > broker lookup > entry_price (DB)"""
    if trade.get("avg_fill_price"):
        return float(trade["avg_fill_price"])

    order_id = trade.get("order_id")
    symbol = trade["symbol"]
    
    if order_id:
        broker_avg = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if broker_avg:
            trades_collection.update_one(
                {"_id": trade["_id"]},
                {"$set": {"avg_fill_price": broker_avg, "entry_price": broker_avg}}
            )
            log.info(f"{symbol}: updated DB with broker avg fill ${broker_avg:.4f}")
            return broker_avg

    log.warning(f"{symbol}: using stored entry_price=${trade['entry_price']:.4f} as fallback")
    return float(trade["entry_price"])


def close_trade(trade: dict, exit_price: float, reason: str):
    symbol = trade["symbol"]
    entry_price = float(trade.get("avg_fill_price") or trade["entry_price"])
    pnl = (exit_price - entry_price) / entry_price

    trades_collection.update_one(
        {"_id": trade["_id"]},
        {"$set": {
            "status": "closed",
            "exit_price": exit_price,
            "exit_reason": reason,
            "exited_at": datetime.now(timezone.utc),
            "pnl_pct": round(pnl * 100, 2),
        }},
    )
    
    log.info(
        f"✓ TRADE CLOSED {symbol} | entry=${entry_price:.4f} | "
        f"exit=${exit_price:.4f} | pnl={pnl*100:.2f}% | reason={reason}"
    )


# ══════════════════════════════════════════════════════════════════════════════
# EMERGENCY EXIT
# ══════════════════════════════════════════════════════════════════════════════

def emergency_exit(ibkr: IBKRSellClient, symbol: str, qty: int, reason: str) -> tuple[float, str]:
    """
    Emergency market exit when speed matters more than price.
    Returns (fill_price, actual_reason).
    """
    log.warning(f"{symbol}: 🚨 EMERGENCY EXIT triggered - {reason}")
    
    # Get current bid
    ticker = ibkr.get_ticker(symbol)
    current_bid = ticker.bid or ticker.last or ticker.close
    
    if not current_bid:
        log.error(f"{symbol}: No price for emergency exit!")
        return 0.0, "exit_failed_no_price"
    
    # Try aggressive limit first (1% below bid)
    aggressive_price = current_bid * 0.99
    trade = ibkr.place_limit_sell(symbol, aggressive_price, qty, emergency=True)
    order_id = trade.order.orderId
    
    # Wait briefly for fill
    for i in range(EMERGENCY_TIMEOUT):
        ibkr.ib.sleep(1)
        fill_price = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if fill_price:
            log.info(f"{symbol}: Emergency limit filled @ ${fill_price:.4f}")
            return fill_price, reason
    
    # Cancel and go market if not filled
    log.warning(f"{symbol}: Limit didn't fill in {EMERGENCY_TIMEOUT}s, switching to MARKET")
    try:
        ibkr.ib.cancelOrder(trade.order)
    except:
        pass
    
    market_trade = ibkr.place_market_sell(symbol, qty)
    ibkr.ib.sleep(2)
    
    fill_price = ibkr.get_avg_fill_price_from_broker(symbol, market_trade.order.orderId)
    if fill_price:
        return fill_price, f"{reason}_market"
    
    # Last resort: use last known price
    return current_bid * 0.98, f"{reason}_estimated"


# ══════════════════════════════════════════════════════════════════════════════
# MAIN MONITOR LOOP
# ══════════════════════════════════════════════════════════════════════════════

def check_exit_conditions(trade: dict, current_price: float, entry_price: float, 
                         stop_level: float, ibkr: IBKRSellClient) -> Optional[str]:
    """
    Check all exit conditions in priority order.
    Returns reason string if should exit, None otherwise.
    """
    symbol = trade["symbol"]
    pnl = (current_price - entry_price) / entry_price
    
    # Priority 1: Hard stop (never lose more than MAX_LOSS_PCT)
    if current_price <= entry_price * (1 - MAX_LOSS_PCT):
        return "hard_stop_max_loss"
    
    # Priority 2: Flash crash detection
    flash_reason = check_flash_crash(symbol, current_price, trade)
    if flash_reason:
        return flash_reason
    
    # Priority 3: Liquidity crisis
    if check_liquidity_crisis(ibkr, symbol):
        return "liquidity_crisis"
    
    # Priority 4: Trailing stop (only if we've made at least 5% profit)
    if trade.get("highest_price", entry_price) >= entry_price * 1.05:
        if current_price <= stop_level:
            return "trailing_stop"
    
    # Priority 5: Profit target
    if pnl >= PROFIT_TARGET:
        return f"profit_target_{pnl*100:.1f}pct"
    
    # Priority 6: PSAR exit signal
    if ibkr.psar_exit_signal(symbol):
        return "psar_exit"
    
    # Priority 7: Original wider stop loss (backup)
    if pnl <= -0.10:  # Your original 10% stop
        return f"stop_loss_{pnl*100:.1f}pct"
    
    return None


def monitor_positions(ibkr: IBKRSellClient):
    log.info("=" * 60)
    log.info("SELL BOT STARTED - Enhanced Loss Protection")
    log.info(f"Hard Stop: {MAX_LOSS_PCT*100}% | Trailing: {TRAILING_STOP_PCT*100}% | Flash: {FLASH_CRASH_PCT*100}%")
    log.info("=" * 60)
    
    while True:
        try:
            open_trades = get_open_trades()
            log.info(f"--- Checking {len(open_trades)} open trade(s) ---")

            for trade in open_trades:
                symbol = trade["symbol"]
                qty = trade["qty"]
                
                # Get entry price
                entry_price = get_effective_entry_price(trade, ibkr)
                
                # Get current market price
                current_price = ibkr.get_last_price(symbol)
                if not current_price:
                    log.warning(f"{symbol}: No current price, skipping")
                    continue
                
                # Update trailing stop and get level
                stop_level = update_trailing_stop(trade, current_price, ibkr)
                
                # Calculate metrics
                pnl = (current_price - entry_price) / entry_price
                highest = trade.get("highest_price", entry_price)
                drawdown_from_high = (current_price - highest) / highest if highest > 0 else 0
                
                # Log status
                log.info(
                    f"{symbol}: entry=${entry_price:.4f} current=${current_price:.4f} "
                    f"pnl={pnl*100:+.2f}% high=${highest:.4f} dd={drawdown_from_high*100:.2f}% "
                    f"stop=${stop_level:.4f}"
                )
                
                # Check all exit conditions
                reason = check_exit_conditions(trade, current_price, entry_price, stop_level, ibkr)
                
                if not reason:
                    continue  # Hold position
                
                # Determine if this is an emergency
                is_emergency = any(x in reason for x in ["flash_crash", "hard_stop", "liquidity"])
                
                if is_emergency:
                    # Emergency exit with market order fallback
                    exit_price, final_reason = emergency_exit(ibkr, symbol, qty, reason)
                    close_trade(trade, exit_price, final_reason)
                else:
                    # Normal limit exit
                    sell_trade = ibkr.place_limit_sell(symbol, current_price, qty)
                    sell_order_id = sell_trade.order.orderId
                    confirmed_exit = ibkr.get_sell_fill_price(symbol, sell_order_id, current_price)
                    close_trade(trade, confirmed_exit, reason)

        except Exception as e:
            log.error(f"Monitor error: {e}", exc_info=True)

        time.sleep(MONITOR_INTERVAL)


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

def main():
    log.info("=== Sell Bot Starting ===")
    ibkr = IBKRSellClient()
    
    try:
        ibkr.connect()
        monitor_positions(ibkr)
    except KeyboardInterrupt:
        log.info("Sell bot stopped by user.")
    except Exception as e:
        log.error(f"Fatal error: {e}", exc_info=True)
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()

2026-03-17 07:43:04,188 [INFO] === Sell Bot Starting ===
2026-03-17 07:43:04,342 [INFO] IBKR connected | accounts: ['DUD087866']
2026-03-17 07:43:04,343 [INFO] ============================================================
2026-03-17 07:43:04,344 [INFO] SELL BOT STARTED - Enhanced Loss Protection
2026-03-17 07:43:04,345 [INFO] Hard Stop: 3.0% | Trailing: 5.0% | Flash: 3.0%
2026-03-17 07:43:04,346 [INFO] ============================================================
2026-03-17 07:43:04,352 [INFO] --- Checking 3 open trade(s) ---
2026-03-17 07:43:15,636 [INFO] KLTR: price source='last' value=$1.3800
2026-03-17 07:43:15,637 [INFO] KLTR: entry=$1.9000 current=$1.3800 pnl=-27.37% high=$1.9000 dd=-27.37% stop=$1.8430
2026-03-17 07:43:15,638 [WARNING] KLTR: 🚨 EMERGENCY EXIT triggered - hard_stop_max_loss


Error 10349, reqId 8: Order TIF was set to GTC based on order preset.
Canceled order: Trade(contract=Stock(conId=479630122, symbol='KLTR', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='KLTR', tradingClass='NMS'), order=LimitOrder(orderId=8, clientId=531, action='SELL', totalQuantity=10, lmtPrice=1.31, outsideRth=True), orderStatus=OrderStatus(orderId=8, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 17, 11, 43, 17, 746805, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 3, 17, 11, 43, 17, 755075, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 8: Order TIF was set to GTC based on order preset.', errorCode=10349)], advancedError='')


2026-03-17 07:43:18,753 [INFO] SELL LIMIT KLTR x10 @ $1.31 orderId=8 emergency=True
2026-03-17 07:43:19,757 [INFO] KLTR: broker avg fill (trades) = $1.3400
2026-03-17 07:43:19,758 [INFO] KLTR: Emergency limit filled @ $1.3400
2026-03-17 07:43:19,760 [INFO] ✓ TRADE CLOSED KLTR | entry=$1.9000 | exit=$1.3400 | pnl=-29.47% | reason=hard_stop_max_loss
2026-03-17 07:43:22,859 [INFO] KLTR: price source='last' value=$1.3800
2026-03-17 07:43:22,860 [INFO] KLTR: entry=$1.8900 current=$1.3800 pnl=-26.98% high=$1.8900 dd=-26.98% stop=$1.8333
2026-03-17 07:43:22,862 [WARNING] KLTR: 🚨 EMERGENCY EXIT triggered - hard_stop_max_loss


Error 10349, reqId 14: Order TIF was set to GTC based on order preset.
Canceled order: Trade(contract=Stock(conId=479630122, symbol='KLTR', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='KLTR', tradingClass='NMS'), order=LimitOrder(orderId=14, clientId=531, action='SELL', totalQuantity=10, lmtPrice=1.31, outsideRth=True), orderStatus=OrderStatus(orderId=14, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 17, 11, 43, 24, 963437, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 3, 17, 11, 43, 24, 969115, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 14: Order TIF was set to GTC based on order preset.', errorCode=10349)], advancedError='')


2026-03-17 07:43:25,971 [INFO] SELL LIMIT KLTR x10 @ $1.31 orderId=14 emergency=True
2026-03-17 07:43:26,985 [INFO] KLTR: broker avg fill (trades) = $1.3400
2026-03-17 07:43:26,986 [INFO] KLTR: Emergency limit filled @ $1.3400
2026-03-17 07:43:26,988 [INFO] ✓ TRADE CLOSED KLTR | entry=$1.8900 | exit=$1.3400 | pnl=-29.10% | reason=hard_stop_max_loss
2026-03-17 07:43:30,144 [INFO] PLBY: price source='last' value=$2.0000
2026-03-17 07:43:30,146 [INFO] PLBY: entry=$2.1100 current=$2.0000 pnl=-5.21% high=$2.1100 dd=-5.21% stop=$2.0467
2026-03-17 07:43:30,148 [WARNING] PLBY: 🚨 EMERGENCY EXIT triggered - hard_stop_max_loss


...

2026-03-17 10:00:23,783 [INFO] SELL LIMIT UCAR x10 @ $1.50 orderId=163 emergency=True
2026-03-17 10:00:24,795 [INFO] UCAR: scanning 7 fill(s) for orderId=163
2026-03-17 10:00:24,796 [WARNING] UCAR: no fill found for orderId=163
2026-03-17 10:00:25,808 [INFO] UCAR: scanning 7 fill(s) for orderId=163
2026-03-17 10:00:25,809 [WARNING] UCAR: no fill found for orderId=163
2026-03-17 10:00:26,813 [INFO] UCAR: scanning 7 fill(s) for orderId=163
2026-03-17 10:00:26,815 [WARNING] UCAR: no fill found for orderId=163
2026-03-17 10:00:27,831 [INFO] UCAR: scanning 7 fill(s) for orderId=163
2026-03-17 10:00:27,832 [WARNING] UCAR: no fill found for orderId=163
2026-03-17 10:00:28,840 [INFO] UCAR: scanning 7 fill(s) for orderId=163
2026-03-17 10:00:28,841 [WARNING] UCAR: no fill found for orderId=163
2026-03-17 10:00:28,842 [WARNING] UCAR: Limit didn't fill in 5s, switching to MARKET


Error 10148, reqId 163: OrderId 163 that needs to be cancelled cannot be cancelled, state: Cancelled.
Error 10349, reqId 165: Order TIF was set to GTC based on order preset.
Canceled order: Trade(contract=Stock(conId=694703219, symbol='UCAR', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='UCAR', tradingClass='SCM'), order=MarketOrder(orderId=165, clientId=531, action='SELL', totalQuantity=10), orderStatus=OrderStatus(orderId=165, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 17, 14, 0, 28, 989172, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 3, 17, 14, 0, 28, 992334, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 165: Order TIF was set to GTC based on order preset.', errorCode=10349)], advan

2026-03-17 10:00:29,994 [INFO] SELL MARKET UCAR x10 orderId=165
2026-03-17 10:00:32,003 [INFO] UCAR: scanning 7 fill(s) for orderId=165
2026-03-17 10:00:32,004 [WARNING] UCAR: no fill found for orderId=165
2026-03-17 10:00:32,008 [INFO] ✓ TRADE CLOSED UCAR | entry=$1.5000 | exit=$1.4994 | pnl=-0.04% | reason=liquidity_crisis_estimated
2026-03-17 10:01:02,011 [INFO] --- Checking 0 open trade(s) ---
2026-03-17 10:01:32,014 [INFO] --- Checking 0 open trade(s) ---
2026-03-17 10:02:02,017 [INFO] --- Checking 0 open trade(s) ---
2026-03-17 10:02:32,020 [INFO] --- Checking 0 open trade(s) ---
2026-03-17 10:03:02,022 [INFO] --- Checking 0 open trade(s) ---
2026-03-17 10:03:32,026 [INFO] --- Checking 0 open trade(s) ---
2026-03-17 10:04:02,029 [INFO] --- Checking 0 open trade(s) ---
2026-03-17 10:04:32,032 [INFO] --- Checking 0 open trade(s) ---
2026-03-17 10:05:02,035 [INFO] --- Checking 0 open trade(s) ---
2026-03-17 10:05:32,038 [INFO] --- Checking 0 open trade(s) ---
2026-03-17 10:06:02,041

[WinError 10054] An existing connection was forcibly closed by the remote host


2026-03-17 10:41:32,263 [ERROR] Monitor error: Socket disconnect
Traceback (most recent call last):
  File "C:\Users\vrajp\anaconda3\Lib\site-packages\ib_insync\util.py", line 341, in run
    result = loop.run_until_complete(task)
  File "C:\Users\vrajp\anaconda3\Lib\site-packages\nest_asyncio.py", line 98, in run_until_complete
    return f.result()
           ~~~~~~~~^^
  File "C:\Users\vrajp\anaconda3\Lib\asyncio\futures.py", line 194, in result
    raise self._make_cancelled_error()
  File "C:\Users\vrajp\anaconda3\Lib\asyncio\tasks.py", line 306, in __step_run_and_handle_result
    result = coro.throw(exc)
  File "C:\Users\vrajp\anaconda3\Lib\site-packages\ib_insync\ib.py", line 1798, in qualifyContractsAsync
    detailsLists = await asyncio.gather(
                   ^^^^^^^^^^^^^^^^^^^^^
        *(self.reqContractDetailsAsync(c) for c in contracts))
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
asyncio.exceptions.CancelledError

During handling of the above exce